PİYASA DENGESİZLİKLERİNİ SİMÜLE EDELİM

Piyasa dengesizliklerini simüle etmek için ajan davranışlarını ve fiyat mekanizmasını daha dinamik hale getirelim. İşte geliştirilmiş model, "Boom-Bust" (Balon-Patlaması) döngüleri, likidite krizleri ve ani şoklar içeren bir simülasyon:

In [ ]:
!pip install mesa[rec]
!pip install seaborn

# Has multi-dimensional arrays and matrices.
# Has a large collection of mathematical functions to operate on these arrays.
import numpy as np

# Data manipulation and analysis.
import pandas as pd

# Data visualization tools.
import seaborn as sns

import mesa

In [ ]:

from mesa.datacollection import DataCollector
import random


class SpekulatorAjan(mesa.Agent):
    def __init__(self, model, risk_alma):
        super().__init__(model)
        self.risk_alma = risk_alma  # 0-1 arası risk iştahı
        self.taraf = self.risk_bazli_secim()

    def risk_bazli_secim(self):
        # Risk iştahı yüksekse talep, düşükse arz tarafında
        return "talep" if random.random() < self.risk_alma else "arz"

    def panik(self, piyasa_volatilite):
        # Volatilite yüksekse %50 olasılıkla taraf değiştir
        if random.random() < piyasa_volatilite * 0.5:
            self.taraf = "arz" if self.taraf == "talep" else "talep"

class DengesizPiyasaModel(mesa.Model):
    def __init__(self, N=100, temel_fiyat=100):
        super().__init__()
        self.temel_fiyat = temel_fiyat
        self.fiyat = temel_fiyat
        self.volatilite = 0
        self.bubble = False

        # Heterojen ajanlar (risk profilleri farklı)
        for i in range(N):
            risk = random.uniform(0.1, 0.9)
            a = SpekulatorAjan(self, risk)
            self.agents.add(a)

        self.datacollector = DataCollector(
            model_reporters={
                "Fiyat": "fiyat",
                "Volatilite": "volatilite",
                "Arz_Talep_Dengesi": "arz_talep_dengesi"
            }
        )

    def arz_talep_dengesi(self):
        talepler = sum(1 for a in self.agents if a.taraf == "talep")
        return talepler / len(self.agents)  # Oran (0-1)

    def fiyat_hesapla(self):
        dengesizlik = self.arz_talep_dengesi() - 0.5  # [-0.5, 0.5]

        # Spekülatif balon etkisi (pozitif geri besleme)
        if self.bubble:
            dengesizlik *= 1.5

        # Fiyat formülü (non-lineer etkiler)
        self.fiyat *= 1 + dengesizlik * 0.1 + random.normalvariate(0, self.volatilite)

        # Volatilite hesapla (fiyat değişimine göre)
        self.volatilite = abs(dengesizlik) * 0.3

        return self.fiyat

    def dis_sok(self, sok_buyukluk):
        # Harici şok (örneğin: finansal kriz, merkez bankası kararı)
        for agent in self.agents:
            if random.random() < 0.7:  # Ajanların %70'i tepki verir
                agent.taraf = "arz"  # Panik satışı

        self.fiyat *= (1 - sok_buyukluk)
        self.volatilite += sok_buyukluk * 2

    def step(self):
        # Balon tespiti (aşırı talep varsa)
        if self.arz_talep_dengesi() > 0.7 and not self.bubble:
            self.bubble = True

        # Balon patlaması
        if self.bubble and random.random() < 0.05:
            self.dis_sok(0.3) # %30 fiyat düşüşü
            self.bubble = False

        # Ajanlar panikleyebilir
        for agent in self.agents:
            agent.panik(self.volatilite)

        self.fiyat = self.fiyat_hesapla()
        self.datacollector.collect(self)

In [ ]:
"""Senaryo Simülasyonları
1. Spekülatif Balon ve Patlama
"""

model = DengesizPiyasaModel(N=200)
for i in range(100):
    if i == 50:  # Balon oluşsun
        for a in model.agents:
            a.taraf = "talep"  # Herkes almaya başlar
    model.step()

In [ ]:
# 2. Likidite Krizi


model = DengesizPiyasaModel(N=100)
for i in range(100):
    if i == 60:
        model.dis_sok(0.4)  # Ani likidite çekilişi
    model.step()

In [ ]:
# 3. Rastgele Şoklar

model = DengesizPiyasaModel(N=150)
for i in range(200):
    if random.random() < 0.02: # %2 olasılıkla rastgele şok
        model.dis_sok(random.uniform(0.1, 0.5))
    model.step()

In [ ]:
# Görselleştirme ve Analiz

data = model.datacollector.get_model_vars_dataframe()

plt.figure(figsize=(12, 8))
plt.subplot(3, 1, 1)
plt.plot(data["Fiyat"], color="blue")
plt.title("Fiyat Dinamikleri (Boom-Bust Döngüleri)")

plt.subplot(3, 1, 2)
plt.plot(data["Volatilite"], color="red")
plt.title("Piyasa Volatilitesi")

plt.subplot(3, 1, 3)
plt.plot(data["Arz_Talep_Dengesi"], color="green")
plt.axhline(y=0.5, linestyle="--", color="gray")
plt.title("Arz-Talep Dengesi (0.5 = Dengeli)")

plt.tight_layout()
plt.show()

